<a href="https://colab.research.google.com/github/Mateo1228/football-match-predictor/blob/main/paso01_explorar_premier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Paso 1 — Abrir y explorar los datos de la Premier League

**Objetivo del paso:** al terminar sabes cuántos partidos tiene una temporada y qué porcentaje gana el local, el visitante y cuántos son empate.

Orden: **1) Intuición → 2) Ejemplo resuelto → 3) Ejercicios tuyos.**

---
## 1. Intuición

Un dataset de partidos es una hoja de Excel donde **cada fila es un partido** y **cada columna es un dato del partido** (fecha, equipos, goles, cuotas…).

En Python esa hoja se llama **DataFrame** y la maneja la librería **pandas**. Casi todo lo que harás en este proyecto son 5 operaciones:

| Si quiero… | Hago… |
|---|---|
| Cargar un CSV | `df = pd.read_csv(url)` |
| Ver cómo se ve | `df.head()`, `df.shape`, `df.columns` |
| Quedarme con unas columnas | `df[['HomeTeam', 'AwayTeam']]` |
| Filtrar filas | `df[df['HomeTeam'] == 'Arsenal']` |
| Contar / promediar | `df['FTR'].value_counts()`, `df['FTHG'].mean()` |

Si quieres reforzar la intuición antes de seguir: la serie **"Pandas Tutorial" de Corey Schafer** en YouTube (los videos 1 y 2 bastan).

### Las columnas que importan (football-data.co.uk)

| Columna | Significa |
|---|---|
| `Date` | Fecha del partido |
| `HomeTeam`, `AwayTeam` | Local y visitante |
| `FTHG`, `FTAG` | Goles al final del partido: local (*Full Time Home Goals*) y visitante |
| `FTR` | Resultado final: `H` = gana local, `D` = empate, `A` = gana visitante |
| `B365H`, `B365D`, `B365A` | Cuotas de Bet365 para local, empate, visitante (las usaremos en el paso 3) |

La lista completa está en https://www.football-data.co.uk/notes.txt

---
## 2. Ejemplo resuelto

Usamos la **temporada 2025/26 completa** (código `2526`, liga `E0` = Premier League). Corre cada celda con Shift + Enter y lee el comentario antes de correrla.

In [94]:
import pandas as pd

# Si quiero cargar una temporada: armo la URL con el código de temporada y de liga
temporada = "2526"   # 2025/26
liga = "E0"          # Premier League (E1 = Championship, SP1 = LaLiga, I1 = Serie A...)
url = f"https://www.football-data.co.uk/mmz4281/{temporada}/{liga}.csv"

df = pd.read_csv(url)
df.head()

,Div,Date,Time,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,...,B365CAHH,B365CAHA,PCAHH,PCAHA,MaxCAHH,MaxCAHA,AvgCAHH,AvgCAHA,BFECAHH,BFECAHA
0,E0,15/08/2025,20:00,Liverpool,Bournemouth,4,2,H,1,0,...,2.03,1.78,2.07,1.85,2.03,1.88,1.94,1.76,2.14,1.86
1,E0,16/08/2025,12:30,Aston Villa,Newcastle,0,0,D,0,0,...,2.05,1.80,2.02,1.89,2.06,1.80,1.95,1.74,2.14,1.86
2,E0,16/08/2025,15:00,Brighton,Fulham,1,1,D,0,0,...,1.83,2.03,1.93,2.00,1.84,2.03,1.80,1.96,1.91,2.08
3,E0,16/08/2025,15:00,Sunderland,West Ham,3,0,H,0,0,...,1.95,1.90,1.97,1.95,1.95,1.94,1.86,1.78,2.02,1.97
4,E0,16/08/2025,15:00,Tottenham,Burnley,3,0,H,1,0,...,1.98,1.88,1.99,1.93,1.98,1.91,1.88,1.83,2.07,1.92


In [95]:
# Si quiero saber el tamaño: shape da (filas, columnas)
print(df.shape)

# Una Premier tiene 20 equipos, cada uno juega contra los otros 19 de local y de visitante:
# 20 * 19 = 380 partidos. Si te da 380 filas, la temporada está completa.
n_partidos = len(df)
print("Partidos:", n_partidos)

(380, 132)
Partidos: 380


In [96]:
# Si quiero quedarme solo con lo útil: selecciono columnas
partidos = df[["Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR"]].copy()
partidos.head(10)

,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR
0,15/08/2025,Liverpool,Bournemouth,4,2,H
1,16/08/2025,Aston Villa,Newcastle,0,0,D
2,16/08/2025,Brighton,Fulham,1,1,D
3,16/08/2025,Sunderland,West Ham,3,0,H
4,16/08/2025,Tottenham,Burnley,3,0,H
5,16/08/2025,Wolves,Man City,0,4,A
6,17/08/2025,Chelsea,Crystal Palace,0,0,D
7,17/08/2025,Nott'm Forest,Brentford,3,1,H
8,17/08/2025,Man United,Arsenal,0,1,A
9,18/08/2025,Leeds,Everton,1,0,H


In [97]:
# Si quiero saber qué tan seguido gana el local:
# 1) cuento cuántas veces aparece cada resultado
conteo = partidos["FTR"].value_counts()
print(conteo)

# 2) lo paso a porcentaje (normalize=True divide por el total)
porcentaje = partidos["FTR"].value_counts(normalize=True) * 100
print(porcentaje.round(1))

FTR
H    162
A    114
D    104
Name: count, dtype: int64
FTR
H    42.6
A    30.0
D    27.4
Name: proportion, dtype: float64


**Cómo leerlo:** si `H` sale alrededor de 40–45 %, eso es la *ventaja de local*. Es el primer patrón que tu modelo tendrá que capturar. Guarda ese número: en el paso 4 será tu **predicción tonta** ("siempre gana el local") que el modelo debe superar.

---
## 3. Ejercicios (tuyos)

Usa la tabla "Si quiero… hago…" de arriba. Cada ejercicio se resuelve con 1–3 líneas.

**Ejercicio 1.** ¿Cuántos goles hubo en promedio por partido (local + visitante)?

*Pista:* crea una columna nueva `partidos["goles_total"] = ...` sumando `FTHG` y `FTAG`, y luego usa `.mean()`.

In [98]:
partidos["goles_total"] = partidos["FTHG"] + partidos["FTAG"]
print (partidos["goles_total"].mean())

2.75


**Ejercicio 2.** ¿Los locales meten más goles que los visitantes? Calcula el promedio de `FTHG` y el de `FTAG` y compáralos.

In [99]:
meanhomegoals = partidos["FTHG"].mean()
meanawaygoals = partidos["FTAG"].mean()

print(f"Local: ", round(meanhomegoals,2),"| Away: ", round(meanawaygoals,2))
print("Difference: ", round(meanhomegoals - meanawaygoals, 2), "goles por partido")



Local:  1.53 | Away:  1.22
Difference:  0.3 goles por partido


**Ejercicio 3.** Filtra solo los partidos de local de tu equipo favorito de la Premier y calcula qué % ganó en casa.

*Pista:* primero filtra con `partidos[partidos["HomeTeam"] == "..."]`, luego repite el `value_counts(normalize=True)` del ejemplo. Si no sabes cómo está escrito el nombre, mira `partidos["HomeTeam"].unique()`.

In [100]:
partidos["HomeTeam"].unique()
es_united = partidos["HomeTeam"] == "Man United"
ManUnited_Local = partidos[es_united]
ManUnited_Local
conteo = ManUnited_Local["FTR"].value_counts(normalize = True)
round(conteo * 100, 2)

,proportion
FTR,
H,68.42
A,15.79
D,15.79


**Ejercicio 4.** ¿Qué equipo metió más goles como local en toda la temporada?

*Pista:* `partidos.groupby("HomeTeam")["FTHG"].sum()` suma los goles por equipo; luego ordénalo con `.sort_values(ascending=False)`.

In [101]:
# Tu código aquí
GolesLocal = partidos.groupby("HomeTeam")["FTHG"].sum()
GolesLocal.sort_values(ascending = False)


,FTHG
HomeTeam,
Man City,45
Arsenal,41
Man United,39
Newcastle,36
Liverpool,34
Brentford,33
Aston Villa,32
Fulham,30
Brighton,30


**Ejercicio 5 (reto).** Carga la temporada actual 2026/27 (código `2627`). ¿Cuántos partidos van? ¿El % de victorias locales se parece al de la temporada pasada?

In [102]:
# Si quiero cargar una temporada: armo la URL con el código de temporada y de liga
temporada = "2627"
liga = "E0"
url = f"https://www.football-data.co.uk/mmz4281/{temporada}/{liga}.csv"
df2026 = pd.read_csv(url)
print(f"Han pasado: {df2026.shape[0]} partidos.")
porcentaje = df2026["FTR"].value_counts(normalize=True)*100
print(porcentaje.round(1))







Han pasado: 80 partidos.
FTR
H    38.8
A    32.5
D    28.7
Name: proportion, dtype: float64


---
### ✅ Terminaste el paso 1 cuando tienes:
- Número de partidos de la temporada
- % de victorias local / empates / visitante
- Goles promedio por partido

Con esos tres números ya puedes hacer tu primer post: *"¿Qué tan grande es la ventaja de local en la Premier?"*